# 1 - Bibliotecas utilizadas

In [1]:
import pandas as pd
from pathlib import Path

# Ignorar avisas
import warnings
warnings.filterwarnings('ignore')

# 2 - Dicionário de Dados 

- `udi`: Identificador único numérico para cada registro na base de dados (varia de 1 a 10.000).
- `id_produto`: Código de identificação exclusivo do produto, composto por uma letra que indica a qualidade e um número de série.
- `tipo`: Categoria do equipamento baseada na sua especificação técnica, sendo dividido em três classes: L (Low - Baixo), M (Medium - Médio) ou H (High - Alto).
- `temperatura_ar_k`: Temperatura do ambiente onde a máquina está instalada, medida na escala Kelvin.
- `temperatura_processo_k`: Temperatura gerada durante a operação do processo de fabricação, medida na escala Kelvin.
- `velocidade_rotacao_rpm`: Velocidade de giro do motor do equipamento, medida em Rotações Por Minuto (RPM).
- `torque_nm`: Força de torção gerada pelo motor da máquina, medida em Newton-metro (Nm).
desgaste_ferramenta_min: Tempo acumulado de utilização da ferramenta de corte atual, medido em minutos.
- `falha_maquina`: Variável Alvo (Target) do projeto. Indica o estado de operação do equipamento, onde 1 representa que a máquina sofreu uma falha mecânica e 0 representa que a máquina operou normalmente.


#### **Nota:** As colunas listadas abaixo representam os motivos técnicos específicos que geraram a quebra da máquina. Elas servem apenas para consulta de histórico e não devem ser utilizadas como variáveis preditoras (X) durante o treinamento dos modelos:

- `falha_twf`: Falha por desgaste da ferramenta (Tool Wear Failure - 1 para sim, 0 para não).
- `falha_hdf`: Falha por dissipação de calor/superaquecimento (Heat Dissipation Failure - 1 para sim, 0 para não).
- `falha_pwf`: Falha por falta ou excesso de potência elétrica (Power Failure - 1 para sim, 0 para não).
- `falha_osf`: Falha por esforço ou tensão mecânica excessiva (Overstrain Failure - 1 para sim, 0 para não).
- `falha_rnf`: Falha aleatória não detectada pelos sensores padrão (Random Failure - 1 para sim, 0 para não).


# 3 - Carregamento e Overview dos dados

In [2]:
# Importando dados do google drive
id_arquivo = '1DVYqjYhbrn5lzUUBtSyGlxn6xikO8n5c'
link = f'https://drive.google.com/uc?id={id_arquivo}'

dados = pd.read_csv(link)

# Salvando dados brutos na camada bronze
#pasta_atual = Path().resolve()
#pasta_raw = pasta_atual.parent / 'data' / '1_bronze'
#pasta_raw.mkdir(parents=True, exist_ok=True)

#caminho_arquivo = pasta_raw / 'manutencao_preditiva.csv'

#dados.to_csv(caminho_arquivo, index=False)

O salvamento dos dados brutos na camada Bronze faz parte do conceito de data lineage (linhagem de dados), ou seja, preserva a origem das informações para garantir a rastreabilidade de todo o ciclo de vida dos dados e de suas transformações para futuras auditorias. O projeto adotará o conceito de arquitetura medalhão:

- Bronze: dados brutos.
- Silver: dados limpos e integrados (sanitizados).
- Gold: dados agregados e prontos para consumo.

In [3]:
# Cópia dos dataframe
df = dados.copy()

In [4]:
# Primeiras 5 linhas
df.head()

,udi,id_produto,tipo,temperatura_ar_k,temperatura_processo_k,velocidade_rotacao_rpm,torque_nm,desgaste_ferramenta_min,falha_maquina,falha_twf,falha_hdf,falha_pwf,falha_osf,falha_rnf
0,1,M14860,M,298.1,308.6,1551.0,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408.0,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498.0,49.4,5,0,0,0,0,0,0
3,4,L47183,L,NaN,NaN,NaN,NaN,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408.0,40.0,9,0,0,0,0,0,0


In [5]:
# Ultimas 5 Linhas
df.tail()

,udi,id_produto,tipo,temperatura_ar_k,temperatura_processo_k,velocidade_rotacao_rpm,torque_nm,desgaste_ferramenta_min,falha_maquina,falha_twf,falha_hdf,falha_pwf,falha_osf,falha_rnf
9995,9996,M24855,M,298.8,308.4,1604.0,29.5,14,0,0,0,0,0,0
9996,9997,H39410,H,298.9,308.4,1632.0,31.8,17,0,0,0,0,0,0
9997,9998,M24857,M,299.0,308.6,1645.0,33.4,22,0,0,0,0,0,0
9998,9999,H39412,H,299.0,308.7,1408.0,48.5,25,0,0,0,0,0,0
9999,10000,M24859,M,299.0,308.7,1500.0,40.2,30,0,0,0,0,0,0


In [6]:
# Shape do dataframe
print(f'Quantidade de linhas: {df.shape[0]}')
print(f'Quantidade de colunas: {df.shape[1]}')

Quantidade de linhas: 10000
Quantidade de colunas: 14


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   udi                      10000 non-null  int64  
 1   id_produto               10000 non-null  object 
 2   tipo                     10000 non-null  object 
 3   temperatura_ar_k         9500 non-null   float64
 4   temperatura_processo_k   9500 non-null   float64
 5   velocidade_rotacao_rpm   9500 non-null   float64
 6   torque_nm                9500 non-null   float64
 7   desgaste_ferramenta_min  10000 non-null  int64  
 8   falha_maquina            10000 non-null  int64  
 9   falha_twf                10000 non-null  int64  
 10  falha_hdf                10000 non-null  int64  
 11  falha_pwf                10000 non-null  int64  
 12  falha_osf                10000 non-null  int64  
 13  falha_rnf                10000 non-null  int64  
dtypes: float64(4), int64(8)

In [8]:
# Verificação de nulos
df.isnull().sum().sort_values(ascending = False)

temperatura_processo_k     500
temperatura_ar_k           500
velocidade_rotacao_rpm     500
torque_nm                  500
id_produto                   0
udi                          0
tipo                         0
desgaste_ferramenta_min      0
falha_maquina                0
falha_twf                    0
falha_hdf                    0
falha_pwf                    0
falha_osf                    0
falha_rnf                    0
dtype: int64

In [9]:
# Verificação de duplicidade do ID
print(f'Total de registros: {df['udi'].value_counts().sum()}')
print (f'Valores únicos variavel "udi": {df['udi'].nunique()}')

Total de registros: 10000
Valores únicos variavel "udi": 10000


## Relatório Overview

A base contém 10.000 registros e 14 variáveis. Não há identificadores duplicados e em principio o tipos de dados estão corretos. Porém contem 500 valores ausentes em cada uma das 4 variáveis abaixo:

- temperatura_processo_k
- temperatura_ar_k
- velocidade_rotacao_rpm
- torque_nm

# 4 - Tratamento de nulos

In [10]:
# Verificando quantidade de registros em que as colunas possuem nulo ao mesmo tempo, caso resultado seja 500
# significa que são sempre ao mesmo tempo
colunas = ['temperatura_ar_k', 'temperatura_processo_k', 'velocidade_rotacao_rpm', 'torque_nm']

print(f'Registros ausentes concomitantes: {df[colunas].isna().all(axis=1).sum()}')

Registros ausentes concomitantes: 500


In [11]:
# Criando um df dos nulos
df_nulos = df[df['temperatura_ar_k'].isnull()]
len(df_nulos)

500

In [12]:
# Primeiras 30 linhas com valores nulos
df_nulos.head(30)

,udi,id_produto,tipo,temperatura_ar_k,temperatura_processo_k,velocidade_rotacao_rpm,torque_nm,desgaste_ferramenta_min,falha_maquina,falha_twf,falha_hdf,falha_pwf,falha_osf,falha_rnf
3,4,L47183,L,NaN,NaN,NaN,NaN,7,0,0,0,0,0,0
14,15,L47194,L,NaN,NaN,NaN,NaN,40,0,0,0,0,0,0
29,30,L47209,L,NaN,NaN,NaN,NaN,84,0,0,0,0,0,0
31,32,L47211,L,NaN,NaN,NaN,NaN,89,0,0,0,0,0,0
33,34,L47213,L,NaN,NaN,NaN,NaN,93,0,0,0,0,0,0
35,36,M14895,M,NaN,NaN,NaN,NaN,98,0,0,0,0,0,0
39,40,L47219,L,NaN,NaN,NaN,NaN,111,0,0,0,0,0,0
76,77,L47256,L,NaN,NaN,NaN,NaN,206,0,0,0,0,0,0
80,81,H29494,H,NaN,NaN,NaN,NaN,4,0,0,0,0,0,0
88,89,M14948,M,NaN,NaN,NaN,NaN,27,0,0,0,0,0,0


In [13]:
# Verificando quantos valores ausente contêm ou não falhas

falha = len(df_nulos[df_nulos['falha_maquina'] == 1])
sem_falha = len(df_nulos[df_nulos['falha_maquina'] == 0])

print(f'Quantidade de valores ausentes com falha: {falha}')
print(f'Quantidade de valores ausentes sem falha: {sem_falha}')
print(f'Proporção dos valores ausentes que contêm falha: {round(falha/500, 2) * 100}% ')

Quantidade de valores ausentes com falha: 17
Quantidade de valores ausentes sem falha: 483
Proporção dos valores ausentes que contêm falha: 3.0% 


In [14]:
len(df[df['temperatura_ar_k'].isnull() & df['temperatura_processo_k'].isnull() & df['velocidade_rotacao_rpm'].isnull() & df['torque_nm'].isnull()])

500

#### Estratégia Adotada

Optei por remover os registros com valores ausentes (5% da base) em vez de imputá-los. Essa decisão se baseia no fato de que os valores nulos ocorrem simultaneamente em 4 das 5 variáveis preditoras. Como apenas 3% desses casos representam falhas, a imputação tem chances minimas de ganho performático para o modelo, aumentaria o custo de processamento e, o mais crítico, introduziria risco de viés (distorção) nos dados.

In [15]:
# Removendo valores ausentes
df_novo = df.dropna()
df_novo

,udi,id_produto,tipo,temperatura_ar_k,temperatura_processo_k,velocidade_rotacao_rpm,torque_nm,desgaste_ferramenta_min,falha_maquina,falha_twf,falha_hdf,falha_pwf,falha_osf,falha_rnf
0,1,M14860,M,298.1,308.6,1551.0,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408.0,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498.0,49.4,5,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408.0,40.0,9,0,0,0,0,0,0
5,6,M14865,M,298.1,308.6,1425.0,41.9,11,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,M24855,M,298.8,308.4,1604.0,29.5,14,0,0,0,0,0,0
9996,9997,H39410,H,298.9,308.4,1632.0,31.8,17,0,0,0,0,0,0
9997,9998,M24857,M,299.0,308.6,1645.0,33.4,22,0,0,0,0,0,0
9998,9999,H39412,H,299.0,308.7,1408.0,48.5,25,0,0,0,0,0,0


In [16]:
df_novo.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9500 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   udi                      9500 non-null   int64  
 1   id_produto               9500 non-null   object 
 2   tipo                     9500 non-null   object 
 3   temperatura_ar_k         9500 non-null   float64
 4   temperatura_processo_k   9500 non-null   float64
 5   velocidade_rotacao_rpm   9500 non-null   float64
 6   torque_nm                9500 non-null   float64
 7   desgaste_ferramenta_min  9500 non-null   int64  
 8   falha_maquina            9500 non-null   int64  
 9   falha_twf                9500 non-null   int64  
 10  falha_hdf                9500 non-null   int64  
 11  falha_pwf                9500 non-null   int64  
 12  falha_osf                9500 non-null   int64  
 13  falha_rnf                9500 non-null   int64  
dtypes: float64(4), int64(8), obje

# 5 - Salvamento na camada Silver

In [17]:
pasta_atual = Path().resolve()
pasta_silver = pasta_atual.parent / 'data' / '2_silver'
pasta_silver.mkdir(parents=True, exist_ok=True)

caminho_silver = pasta_silver / 'manutencao_preditiva-silver.csv'

df_novo.to_csv(caminho_silver, index=False)

# 6 - Proximos passos

Avançaremos agora para as fases de preparação e modelagem do projeto:

- `Análise Exploratória` e Engenharia de Features: Mapear padrões e criar variáveis preditivas para facilitar o aprendizado do algoritmo.

- `Modelagem Preditiva`: Treinar e validar o modelo focado na identificação antecipada de falhas.